# 08 · Experiment E3: which encoder, and at what cost? (task T24, runs R16 to R21)

**RQ3.** With only 252 training chips, is a light pretrained encoder better than a small network trained from nothing? Everything else is the frozen default (VV + VH + ratio input, BCE + Dice loss):

| Config | Encoder | Runs |
|---|---|---|
| Scratch U-Net | 5 levels, 16–256 filters, random initial weights | R16, R17, R18 |
| Default | MobileNetV2, random initial weights | R01, R02, R03 (shared) |
| MobileNetV2 + ImageNet | MobileNetV2 starting from ImageNet weights | R19, R20, R21 |

ImageNet weights were learned on ordinary colour photographs scaled to [−1, 1]. The standardised radar channels look nothing like that, so whether the weights still help is part of the question.

The second half of the notebook measures what each encoder **costs**: parameters, time per training step, time per predicted chip and peak GPU memory.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT))
import src  # noqa: F401  (must come before TensorFlow: switches on the XLA compile cache)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import analysis, viz
from src.config import STUDIES, load_config, run_table, runs_of
from src.runner import check_frozen, run_benchmark, run_evaluation, run_many

viz.use_style()
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
print("training and scoring code is identical to the frozen tag 'defaults-frozen' (commit %s)" % check_frozen()[:7])

STUDY = "E3"
CONFIGS = STUDIES[STUDY]["configs"]
pd.DataFrame([r for r in run_table() if r["config_name"] in CONFIGS]).fillna({"encoder": "mobilenetv2"}).set_index("run_id")

## 1. Train and score R16 to R21

In [ ]:
results = run_many(["R16", "R17", "R18", "R19", "R20", "R21"])

## 2. Results

One row per run. Validation IoU is the score the checkpoint was chosen by; test and Bolivia were scored once, after training.

In [ ]:
runs = analysis.runs_table()
study_runs = runs[runs["Config"].isin(CONFIGS)]
study_runs[["Run ID", "Config", "Seed", "Status", "Epochs trained", "Best epoch", "Val IoU (best)", "Test IoU",
            "Test precision", "Test recall", "Bolivia IoU", "Sec per epoch"]].set_index("Run ID")

Mean ± standard deviation over the three seeds (the two threshold baselines, single deterministic runs, are shown for reference):

In [ ]:
analysis.study_table(STUDY, analysis.summary_table(runs))

**Is a difference real?** The project's rule: a gap between two configs counts only if it is larger than the seed-to-seed standard deviation.

In [ ]:
gaps = analysis.gap_table(STUDY, runs)
gaps.to_csv(ROOT / "results" / f"{STUDY}_gaps.csv", index=False)
gaps.set_index(["Metric", "A", "B"])

In [ ]:
hist = {c: analysis.histories(runs_of(c)) for c in CONFIGS}
axes = viz.plot_val_curves(hist)
axes[0].figure.tight_layout()
viz.savefig(axes[0].figure, ROOT / "figures" / f"{STUDY}_val_curves.png")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.3), sharey=True)
done = runs[runs["Status"] == "Finished"]
for ax, metric in zip(axes, ("Test IoU", "Bolivia IoU")):
    viz.plot_config_bars(done, CONFIGS, metric, ax, baseline=("Otsu (E0)", "Tuned VH threshold"))
    ax.set_title(metric.replace("IoU", "water IoU"))
fig.tight_layout()
viz.savefig(fig, ROOT / "figures" / f"{STUDY}_bars.png")
plt.show()

## 3. Cost of each encoder

Two sources:

* **logged by the real runs** (mean over the three seeds): epochs needed, seconds per epoch including validation, peak GPU memory and peak host RAM of the training process;
* **a controlled benchmark** (`src/benchmark.py`): each model alone on the GPU, 30 training steps on one fixed batch of 8 crops (256×256) and 30 predictions of 8 full chips (512×512), medians. The logged seconds per epoch depend on how warm and busy the laptop was during a run; the benchmark times all three models the same way.

In [ ]:
bench_file = ROOT / "results" / "cost_benchmark.csv"
if bench_file.exists():
    bench = pd.read_csv(bench_file, index_col=0)
else:
    bench = pd.DataFrame({c: run_benchmark(encoder=load_config(runs_of(c)[0])["encoder"]) for c in CONFIGS}).T
    bench.index.name = "Config"
    bench.to_csv(bench_file)

logged = done[done["Config"].isin(CONFIGS)].groupby("Config")[
    ["Epochs trained", "Sec per epoch", "Train minutes", "Peak GPU memory (MB)", "Peak host RAM (MB)"]].mean()
cost = pd.DataFrame({
    "Parameters (millions)": bench["total_params"].astype(float) / 1e6,
    "Training step (ms), benchmark": bench["train_step_ms"].astype(float),
    "Inference per chip (ms), benchmark": bench["inference_ms_per_chip"].astype(float),
    "Peak GPU memory, training (MB), benchmark": bench["peak_gpu_memory_train_mb"].astype(float),
    "Peak GPU memory, inference (MB), benchmark": bench["peak_gpu_memory_inference_mb"].astype(float),
    "Epochs trained, mean of 3 runs": logged["Epochs trained"],
    "Seconds per epoch, mean of 3 runs": logged["Sec per epoch"],
    "Training minutes per run, mean of 3 runs": logged["Train minutes"],
    "Peak GPU memory incl. validation (MB), mean of 3 runs": logged["Peak GPU memory (MB)"],
    "Peak host RAM (MB), mean of 3 runs": logged["Peak host RAM (MB)"],
}).loc[list(CONFIGS)]
cost.to_csv(ROOT / "results" / "cost_table.csv")
cost.T.round(2)